# Experiment: Combined Berthoud Residual U-Net V2

Train the residual U-Net on the combined Berthoud V2 dataset: original HRRR pairs, salvaged Oct-Dec HRRR pairs, and controlled speed/direction mass/momentum pairs.


## Drive Inputs

Upload these files to `MyDrive/windninja_ml/` before running this notebook:

- `residual_unet_code.zip`
- `berthoud_combined_v2_dataset.zip`

The notebook trains from local Colab disk and writes checkpoints, logs, and evaluation outputs back to Drive.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = Path('/content').exists()
DRIVE_ROOT = Path('/content/drive/MyDrive/windninja_ml') if IN_COLAB else Path.cwd() / 'ml/residual_unet/outputs/colab_local'
REPO_DIR = Path('/content/mountain_windninja') if IN_COLAB else Path.cwd()
LOCAL_DATA_ROOT = Path('/content/data') if IN_COLAB else REPO_DIR / 'ml/residual_unet/data/processed'
DATASET_NAME = 'berthoud_combined_v2'
LOCAL_DATA = LOCAL_DATA_ROOT / DATASET_NAME
CODE_ZIP = DRIVE_ROOT / 'residual_unet_code.zip'
DATASET_ZIP = DRIVE_ROOT / f'{DATASET_NAME}_dataset.zip'
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / DATASET_NAME
LOG_CSV = DRIVE_ROOT / 'logs' / f'{DATASET_NAME}_train_log.csv'
EVAL_ROOT = DRIVE_ROOT / 'eval' / DATASET_NAME

REPO_DIR, LOCAL_DATA, CHECKPOINT_DIR


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV.parent.mkdir(parents=True, exist_ok=True)
EVAL_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')


## Unpack Code And Data

This intentionally replaces the temporary `/content` copy on each run. Persistent outputs remain in Drive.


In [ ]:
if IN_COLAB:
    if not CODE_ZIP.exists():
        raise FileNotFoundError(f'Missing code ZIP: {CODE_ZIP}')
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f'Missing dataset ZIP: {DATASET_ZIP}')
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP) as archive:
        archive.extractall(REPO_DIR)
    if LOCAL_DATA.exists():
        shutil.rmtree(LOCAL_DATA)
    LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as archive:
        archive.extractall(LOCAL_DATA_ROOT)

assert (REPO_DIR / 'ml/residual_unet/configs/berthoud_combined_v2.yaml').exists(), REPO_DIR
assert (LOCAL_DATA / 'manifest.csv').exists(), LOCAL_DATA
assert (LOCAL_DATA / 'normalization.json').exists(), LOCAL_DATA
summary = json.loads((LOCAL_DATA / 'dataset_summary.json').read_text())
print(json.dumps(summary, indent=2))


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'ml/residual_unet/requirements.txt')],
    check=True,
)


## Train

The configured run uses 80 epochs. If Colab disconnects, rerun from the top; this cell resumes from `latest.pt` when present.


In [ ]:
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
cmd = [
    sys.executable,
    '-m',
    'ml.residual_unet.train',
    '--config',
    str(REPO_DIR / 'ml/residual_unet/configs/berthoud_combined_v2.yaml'),
    '--data',
    str(LOCAL_DATA),
    '--checkpoint-dir',
    str(CHECKPOINT_DIR),
    '--log-csv',
    str(LOG_CSV),
]
resume = CHECKPOINT_DIR / 'latest.pt'
if resume.exists():
    cmd += ['--resume', str(resume)]
subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)


## Evaluate

This writes one combined test report plus separate reports for HRRR-derived and controlled held-out cases.


In [ ]:
eval_jobs = [
    ("all", None),
    ("hrrr_berthoud_v0", "berthoud_v0"),
    ("hrrr_oct_dec_2025", "berthoud_hrrr_oct_dec_2025_v1"),
    ("controlled_berthoud_training", "controlled_berthoud_training"),
]
results = {}
for label, source_dataset in eval_jobs:
    out_dir = EVAL_ROOT / label
    cmd = [
        sys.executable,
        "-m",
        "ml.residual_unet.evaluate",
        "--checkpoint",
        str(CHECKPOINT_DIR / "best.pt"),
        "--data",
        str(LOCAL_DATA),
        "--out",
        str(out_dir),
        "--split",
        "test",
    ]
    if source_dataset:
        cmd += ["--source-dataset", source_dataset]
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    results[label] = json.loads((out_dir / "metrics.json").read_text())
results
